In [7]:
# ---------------------------
# 0. 설치/마운트
# ---------------------------
# (Colab에서만 실행)
!pip -q install ultralytics==8.0.0 opencv-python-headless scikit-learn pandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 219.8/219.8 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 59.7 MB/s eta 0:00:00


In [2]:
# ============================================================
# 정면 5. 상추단위 Seg → 4개 선발 → 저장/로그 (Colab용)
# - 재귀 탐색
# - chunk 처리 + 중간저장(재시작)
# - tqdm 진행률/ETA
# - ThreadPool 병렬(저장/후처리)
# - Drive → /content 로 zip 복사(빠른 I/O)
# - 원본 crop 해상도 유지
#
# 요구사항 반영:
# 0) Drive 지정 폴더를 /content로 옮김(zip)
# 1) 2,192장 seg 추론 + crop 후보 생성
# 2) 규칙에 따라 4개(또는 <=4 전부) 선발하여 저장
# 3) 저장 결과 로그(CSV) + manifest(재시작)
# 4) 결과를 zip으로 Drive로 이동
# ============================================================

from __future__ import annotations
import os
import re
import cv2
import json
import time
import math
import shutil #파일과 폴더를 복사, 이동, 삭제하는 도구
import zipfile
import hashlib #중복 방지를 위해 씁니다
from dataclasses import dataclass
from pathlib import Path
from typing import List, Dict, Tuple, Optional
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from sklearn.cluster import KMeans

# Ultralytics YOLO (seg)
from ultralytics import YOLO

# ---------------------------
# 1. 사용자 설정(여기만 수정)
# ---------------------------
@dataclass
class CFG:
    # Drive 경로
    DRIVE_INPUT_DIR: str = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/ROI_WARP/step6_warp_images/보정 후"  # 베드 이미지 폴더(재귀탐색)
    DRIVE_OUTPUT_DIR: str = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/lettuce"  # 최종 결과 저장 폴더

    # YOLO Seg 모델 경로(.pt)
    SEG_MODEL_PATH: str = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/model/results/runs/bed_seg_v1/weights/best.pt"

    # 입력 이미지 확장자
    IMG_EXTS: Tuple[str, ...] = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")

    # /content 작업 폴더
    WORK_DIR: str = "/content/_front5_work"

    # 추론 파라미터
    CONF_THRES: float = 0.2
    IOU_THRES: float = 0.7
    MAX_DET: int = 300

    # Crop 옵션
    CROP_PAD: int = 6          # bbox 패딩(px)
    MIN_AREA_PX: int = 150     # 너무 작은 잡검출 제거(면적)

    # Row split 옵션
    USE_KMEANS_ROW_SPLIT: bool = True
    Y_SPREAD_MIN: float = 10.0   # y가 너무 비슷하면(상하 분리 불가) 한 행 처리

    # 중앙 기준
    CENTER_X_MODE: str = "image"  # "image" or "median" (row별)

    # 저장/처리
    CHUNK_SIZE: int = 200
    NUM_WORKERS: int = 8

    # 재시작/중간저장
    RESUME: bool = True

cfg = CFG()


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:

# ---------------------------
# 2. 유틸
# ---------------------------

def ensure_dir(p: str | Path) -> Path:
    p = Path(p)
    p.mkdir(parents=True, exist_ok=True)
    return p


def list_images_recursive(root: str | Path, exts: Tuple[str, ...]) -> List[Path]:
    root = Path(root)
    out = []
    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in exts:
            out.append(p)
    out.sort()
    return out


def zip_dir(src_dir: Path, zip_path: Path) -> None:
    src_dir = src_dir.resolve()
    zip_path = zip_path.resolve()
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for f in src_dir.rglob("*"):
            if f.is_file():
                arcname = f.relative_to(src_dir)
                zf.write(f, arcname.as_posix())


def unzip_to(zip_path: Path, dst_dir: Path) -> None:
    if dst_dir.exists():
        shutil.rmtree(dst_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(dst_dir)


def safe_stem(p: Path) -> str:
    # 파일명에서 확장자 제거
    return p.stem


def clamp(v: int, lo: int, hi: int) -> int:
    return max(lo, min(hi, v))


def bbox_from_poly(poly_xy: np.ndarray) -> Tuple[int, int, int, int]:
    # poly_xy: (N,2)
    xs = poly_xy[:, 0]
    ys = poly_xy[:, 1]
    x1 = int(np.floor(xs.min()))
    y1 = int(np.floor(ys.min()))
    x2 = int(np.ceil(xs.max()))
    y2 = int(np.ceil(ys.max()))
    return x1, y1, x2, y2


def centroid_from_poly(poly_xy: np.ndarray) -> Tuple[float, float]:
    # 폴리곤 중심(단순 평균; 안정성 충분)
    cx = float(np.mean(poly_xy[:, 0]))
    cy = float(np.mean(poly_xy[:, 1]))
    return cx, cy


def crop_by_bbox(img: np.ndarray, bbox: Tuple[int, int, int, int], pad: int = 0) -> np.ndarray:
    h, w = img.shape[:2]
    x1, y1, x2, y2 = bbox
    x1 = clamp(x1 - pad, 0, w - 1)
    y1 = clamp(y1 - pad, 0, h - 1)
    x2 = clamp(x2 + pad, 0, w - 1)
    y2 = clamp(y2 + pad, 0, h - 1)
    if x2 <= x1 or y2 <= y1:
        return np.zeros((1, 1, 3), dtype=img.dtype)
    return img[y1:y2, x1:x2].copy()


# ---------------------------
# 3. 4개 선발 로직
# ---------------------------

def split_rows_by_y(cys: np.ndarray, y_spread_min: float, use_kmeans: bool = True) -> np.ndarray:
    """return row labels: 0(top), 1(bottom) or all 0 if split not reliable"""
    if len(cys) < 2:
        return np.zeros(len(cys), dtype=int)
    if (cys.max() - cys.min()) < y_spread_min:
        return np.zeros(len(cys), dtype=int)

    if use_kmeans:
        km = KMeans(n_clusters=2, n_init=10, random_state=42)
        labels = km.fit_predict(cys.reshape(-1, 1))
        # labels 0/1을 top/bottom으로 정렬
        centers = km.cluster_centers_.reshape(-1)
        top_label = int(np.argmin(centers))
        out = np.where(labels == top_label, 0, 1)
        return out.astype(int)
    else:
        # y 기준 threshold(중앙값)
        thr = float(np.median(cys))
        out = (cys > thr).astype(int)  # 0 top, 1 bottom
        return out


def pick_central_k(indices_sorted_by_x: List[int], xs: np.ndarray, k: int, x0: float) -> List[int]:
    """indices_sorted_by_x: row 내 인덱스(원본 인덱스) 리스트. k개를 x0 기준으로 가장 가까운 것들로 선택"""
    if len(indices_sorted_by_x) <= k:
        return indices_sorted_by_x
    # 거리 기준으로 k개 선택, 선택 후 x순서 유지
    d = [(abs(xs[i] - x0), i) for i in indices_sorted_by_x]
    d.sort(key=lambda t: t[0])
    chosen = sorted([i for _, i in d[:k]], key=lambda i: xs[i])
    return chosen


def select_instances(xs: np.ndarray, ys: np.ndarray, img_w: int) -> Tuple[List[int], str, Dict]:
    """규칙에 따라 최종 선택 인덱스 반환"""
    n = len(xs)
    meta = {}

    if n == 0:
        return [], "no_detect", meta

    # <=4면 전부
    if n <= 4:
        return list(range(n)), "n_le_4", meta

    # row split
    row = split_rows_by_y(ys, cfg.Y_SPREAD_MIN, cfg.USE_KMEANS_ROW_SPLIT)
    top_idx = [i for i in range(n) if row[i] == 0]
    bot_idx = [i for i in range(n) if row[i] == 1]

    # 각 row x정렬
    top_idx = sorted(top_idx, key=lambda i: xs[i])
    bot_idx = sorted(bot_idx, key=lambda i: xs[i])

    meta.update({"n_top": len(top_idx), "n_bot": len(bot_idx)})

    # 중앙 기준 x0
    if cfg.CENTER_X_MODE == "median":
        x0 = float(np.median(xs))
    else:
        x0 = float(img_w / 2.0)

    # 한 행만 사실상 존재
    if len(top_idx) == 0 and len(bot_idx) > 0:
        chosen = pick_central_k(bot_idx, xs, 4, x0)
        return chosen, "only_bottom", meta
    if len(bot_idx) == 0 and len(top_idx) > 0:
        chosen = pick_central_k(top_idx, xs, 4, x0)
        return chosen, "only_top", meta

    # 둘 다 존재: 기본은 위2 + 아래2(중앙기준)
    top_ch = pick_central_k(top_idx, xs, 2, x0)
    bot_ch = pick_central_k(bot_idx, xs, 2, x0)

    # 만약 한쪽이 2개 미만이면(위1 아래5 등) -> 있는 만큼 쓰고 나머지는 다른 쪽에서 중앙으로 채움
    chosen = []
    chosen.extend(top_ch)
    chosen.extend(bot_ch)

    if len(chosen) < 4:
        remain = 4 - len(chosen)
        # 후보는 chosen에 없는 것들 중에서, 다른 row 전체에서 중앙 가까운 순으로
        pool = [i for i in range(n) if i not in chosen]
        pool_sorted = sorted(pool, key=lambda i: abs(xs[i] - x0))
        fill = pool_sorted[:remain]
        # 최종은 x순서로 정렬(저장 순서 안정)
        chosen = sorted(chosen + fill, key=lambda i: xs[i])
        return chosen, "imbalanced_fill", meta

    # 최종 4개는 (top/bot 각각 x순서 유지)
    # 다만 합쳐질 때 top/bot 섞이지 않게 저장명에서 row/col을 표현할 거라 순서는 상관 적음
    chosen = top_ch + bot_ch
    return chosen, "top2_bot2", meta


def assign_slot_names(xs: np.ndarray, ys: np.ndarray, chosen_idx: List[int]) -> Dict[int, str]:
    """선택된 인덱스에 대해 t/b + col을 부여. col은 해당 row 내 x정렬에서의 원래 위치(1-based)"""
    if len(chosen_idx) == 0:
        return {}

    # row 재계산(선택/비선택 상관 없이 전체 기준)
    row = split_rows_by_y(ys, cfg.Y_SPREAD_MIN, cfg.USE_KMEANS_ROW_SPLIT)
    top_all = [i for i in range(len(xs)) if row[i] == 0]
    bot_all = [i for i in range(len(xs)) if row[i] == 1]
    top_all = sorted(top_all, key=lambda i: xs[i])
    bot_all = sorted(bot_all, key=lambda i: xs[i])

    top_pos = {i: (j + 1) for j, i in enumerate(top_all)}
    bot_pos = {i: (j + 1) for j, i in enumerate(bot_all)}

    slot = {}
    for i in chosen_idx:
        if row[i] == 1:
            slot[i] = f"b{bot_pos.get(i, 0)}"
        else:
            slot[i] = f"t{top_pos.get(i, 0)}"

    # row 분리 실패(전부 top=0)인 경우도 있음 -> t만 나오게 됨. 그건 의도대로 OK.
    return slot



In [4]:

# ---------------------------
# 4. 단일 이미지 처리
# ---------------------------

def process_one_image(model: YOLO, img_path: Path, out_root: Path) -> Dict:
    """한 이미지에 대해: 추론 → 후보 bbox/centroid → 선발 → crop 저장"""
    t0 = time.time()

    img = cv2.imread(str(img_path))
    if img is None:
        return {"image": str(img_path), "status": "read_fail"}

    h, w = img.shape[:2]

    # 추론
    results = model.predict(
        source=img,
        conf=cfg.CONF_THRES,
        iou=cfg.IOU_THRES,
        max_det=cfg.MAX_DET,
        verbose=False
    )
    r = results[0]

    if r.masks is None or r.masks.xy is None or len(r.masks.xy) == 0:
        return {
            "image": str(img_path),
            "status": "no_mask",
            "n_detect": 0,
            "elapsed_sec": round(time.time() - t0, 3)
        }

    polys = r.masks.xy  # list of (N,2) float arrays

    items = []
    for poly in polys:
        poly = np.asarray(poly, dtype=np.float32)
        if poly.ndim != 2 or poly.shape[0] < 3:
            continue
        x1, y1, x2, y2 = bbox_from_poly(poly)
        area = max(0, (x2 - x1)) * max(0, (y2 - y1))
        if area < cfg.MIN_AREA_PX:
            continue
        cx, cy = centroid_from_poly(poly)
        items.append({
            "bbox": (x1, y1, x2, y2),
            "cx": float(cx),
            "cy": float(cy),
            "area": int(area)
        })

    if len(items) == 0:
        return {
            "image": str(img_path),
            "status": "filtered_all",
            "n_detect": 0,
            "elapsed_sec": round(time.time() - t0, 3)
        }

    xs = np.array([it["cx"] for it in items], dtype=np.float32)
    ys = np.array([it["cy"] for it in items], dtype=np.float32)

    chosen, rule, meta = select_instances(xs, ys, img_w=w)

    # 저장 폴더
    stem = safe_stem(img_path)
    out_dir = out_root / stem
    out_dir.mkdir(parents=True, exist_ok=True)

    slot_map = assign_slot_names(xs, ys, chosen)

    saved = []
    for idx in chosen:
        crop = crop_by_bbox(img, items[idx]["bbox"], pad=cfg.CROP_PAD)
        slot = slot_map.get(idx, f"u{idx+1}")
        out_name = f"{stem}_{slot}.jpg"
        out_path = out_dir / out_name
        ok = cv2.imwrite(str(out_path), crop)
        saved.append({"idx": idx, "slot": slot, "path": str(out_path), "ok": bool(ok)})

    elapsed = time.time() - t0

    return {
        "image": str(img_path),
        "status": "ok" if len(saved) > 0 else "no_save",
        "n_detect": int(len(items)),
        "n_saved": int(len(saved)),
        "rule": rule,
        "n_top": int(meta.get("n_top", -1)),
        "n_bot": int(meta.get("n_bot", -1)),
        "saved_slots": ";".join([s["slot"] for s in saved if s["ok"]]),
        "elapsed_sec": round(elapsed, 3),
    }


In [11]:

# ---------------------------
# 5. 파이프라인 실행(Drive→content, chunk, resume, zip back)
# ---------------------------

def main():
    work = ensure_dir(cfg.WORK_DIR)
    tmp_in_zip = work / "input.zip"
    tmp_in_dir = work / "input"
    tmp_out_dir = ensure_dir(work / "out")

    ensure_dir(Path(cfg.DRIVE_OUTPUT_DIR))

    # 5-2) Drive 입력을 /content로 zip→unzip(빠른 IO)
    src_dir = Path(cfg.DRIVE_INPUT_DIR)
    if not src_dir.exists():
        raise FileNotFoundError(f"입력 폴더가 없음: {src_dir}")

    print("[0] Drive → /content : zip 생성")
    zip_dir(src_dir, tmp_in_zip)

    print("[0] unzip to /content")
    unzip_to(tmp_in_zip, tmp_in_dir)

    # 5-3) 이미지 목록
    img_list = list_images_recursive(tmp_in_dir, cfg.IMG_EXTS)
    print(f"총 이미지: {len(img_list)}")

    # 5-4) resume/manifest
    manifest_path = work / "manifest.csv"
    log_path = work / "save_log.csv"

    done_set = set()
    if cfg.RESUME and manifest_path.exists():
        try:
            mdf = pd.read_csv(manifest_path)
            done_set = set(mdf.loc[mdf["done"] == 1, "image"].astype(str).tolist())
            print(f"RESUME: 이미 처리완료 {len(done_set)}개")
        except Exception as e:
            print("manifest 읽기 실패, 새로 시작:", e)

    # 로그 파일 헤더 준비
    if not log_path.exists():
        pd.DataFrame(columns=[
            "image","status","n_detect","n_saved","rule","n_top","n_bot","saved_slots","elapsed_sec"
        ]).to_csv(log_path, index=False)

    if not manifest_path.exists():
        pd.DataFrame(columns=["image","done"]).to_csv(manifest_path, index=False)

    # 5-5) 모델 로드
    print("[1] 모델 로드")
    model = YOLO(cfg.SEG_MODEL_PATH)

    # 5-6) 처리 대상
    targets = [p for p in img_list if str(p) not in done_set]
    print(f"남은 처리 대상: {len(targets)}")

    # 5-7) chunk 처리
    def append_rows_csv(csv_path: Path, rows: List[Dict]):
        if not rows:
            return
        df = pd.DataFrame(rows)
        df.to_csv(csv_path, mode="a", header=False, index=False)

    def update_manifest(manifest_path: Path, images_done: List[str]):
        if not images_done:
            return
        # 기존 + 추가를 합쳐 중복 제거
        try:
            m = pd.read_csv(manifest_path)
        except Exception:
            m = pd.DataFrame(columns=["image","done"])
        add = pd.DataFrame({"image": images_done, "done": 1})
        m2 = pd.concat([m, add], ignore_index=True)
        m2 = m2.drop_duplicates(subset=["image"], keep="last")
        m2.to_csv(manifest_path, index=False)

    chunks = [targets[i:i+cfg.CHUNK_SIZE] for i in range(0, len(targets), cfg.CHUNK_SIZE)]

    print(f"CHUNK: {len(chunks)}개 (chunk_size={cfg.CHUNK_SIZE})")

    for ci, chunk in enumerate(chunks, start=1):
        pbar = tqdm(total=len(chunk), desc=f"Chunk {ci}/{len(chunks)}")
        rows = []
        done_imgs = []

        # 병렬: 저장/후처리 IO가 많아서 Thread가 효율적
        with ThreadPoolExecutor(max_workers=cfg.NUM_WORKERS) as ex:
            futures = [ex.submit(process_one_image, model, img_path, tmp_out_dir) for img_path in chunk]
            for fut in as_completed(futures):
                res = fut.result()
                rows.append(res)
                done_imgs.append(res["image"])
                pbar.update(1)
        pbar.close()

        # 로그/manifest flush
        append_rows_csv(log_path, rows)
        update_manifest(manifest_path, done_imgs)

        # chunk마다 메모리/캐시 정리(큰 효과는 없지만 안정성)
        del rows

    # 5-8) 결과 zip 생성
    out_zip = work / "lettuce_crops_out.zip"
    print("[4] 결과 zip 생성")
    if out_zip.exists():
        out_zip.unlink()
    with zipfile.ZipFile(out_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for f in tmp_out_dir.rglob("*"):
            if f.is_file():
                zf.write(f, f.relative_to(tmp_out_dir).as_posix())

    # 5-9) Drive로 이동
    dst_dir = Path(cfg.DRIVE_OUTPUT_DIR)
    dst_dir.mkdir(parents=True, exist_ok=True)
    dst_zip = dst_dir / out_zip.name
    print(f"[4] Drive로 복사: {dst_zip}")
    shutil.copy2(out_zip, dst_zip)

    # 로그/manifest도 Drive에 복사
    shutil.copy2(log_path, dst_dir / log_path.name)
    shutil.copy2(manifest_path, dst_dir / manifest_path.name)

    print("완료!")
    print("- 결과 zip:", dst_zip)
    print("- 로그:", dst_dir / log_path.name)
    print("- manifest:", dst_dir / manifest_path.name)


In [12]:
# ---------------------------
# 6. 실행
# ---------------------------
main()

[0] Drive → /content : zip 생성
[0] unzip to /content
총 이미지: 2192
RESUME: 이미 처리완료 0개
[1] 모델 로드
남은 처리 대상: 2192
CHUNK: 11개 (chunk_size=200)


Chunk 1/11:   0%|          | 0/200 [00:00<?, ?it/s]

Ultralytics 8.4.7 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (NVIDIA L4, 22693MiB)
Ultralytics 8.4.7 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (NVIDIA L4, 22693MiB)
Ultralytics 8.4.7 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (NVIDIA L4, 22693MiB)
Ultralytics 8.4.7 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (NVIDIA L4, 22693MiB)
Ultralytics 8.4.7 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (NVIDIA L4, 22693MiB)
Ultralytics 8.4.7 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (NVIDIA L4, 22693MiB)
Ultralytics 8.4.7 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (NVIDIA L4, 22693MiB)
YOLOv8n-seg summary (fused): 86 layers, 3,258,259 parameters, 0 gradients, 11.3 GFLOPs
YOLOv8n-seg summary (fused): 86 layers, 3,258,259 parameters, 0 gradients, 11.3 GFLOPs
YOLOv8n-seg summary (fused): 86 layers, 3,258,259 parameters, 0 gradients, 11.3 GFLOPs
YOLOv8n-seg summary (fused): 86 layers, 3,258,259 parameters, 0 gradients, 11.3 GFLOPs
YOLOv8n-seg summary (fused): 86 layers, 3,258,259 parameters, 0 gradients, 11.

Chunk 2/11:   0%|          | 0/200 [00:00<?, ?it/s]

Chunk 3/11:   0%|          | 0/200 [00:00<?, ?it/s]

...

Chunk 8/11:   0%|          | 0/200 [00:00<?, ?it/s]

Chunk 9/11:   0%|          | 0/200 [00:00<?, ?it/s]

Chunk 10/11:   0%|          | 0/200 [00:00<?, ?it/s]

Chunk 11/11:   0%|          | 0/192 [00:00<?, ?it/s]

[4] 결과 zip 생성
[4] Drive로 복사: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/lettuce/lettuce_crops_out.zip
완료!
- 결과 zip: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/lettuce/lettuce_crops_out.zip
- 로그: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/lettuce/save_log.csv
- manifest: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/lettuce/manifest.csv
